# Camera Discovery — All-media Harvest Handoff to Full Validation

This notebook tests harvest mode with all supported media types, then passes the harvest handoff into the normal pipeline. Harvest mode can collect `hls`, `mjpeg`, `image_snapshot`, `video_file`, `stream`, and `unknown_media`; the normal pipeline may only have first-class validation/trust behavior for its supported camera candidate types.

Use this notebook to inspect how broader harvested media is represented, scoped, validated, trusted, or left for review. It can generate larger artifacts than the HLS-only notebook.


## Setup

This notebook installs repository code from the configured Git branch and runs the public CLI. It does not patch source files from the notebook.

Default behavior clones the `dev` branch. In Colab, restart the runtime after dependency installation if package imports behave unexpectedly.


In [ ]:
# Repository setup for Google Colab / notebook execution.
# Change REPO_BRANCH or REPO_URL if testing a fork/PR branch.
REPO_BRANCH = "dev"
REPO_URL = "https://github.com/dshipley71/camera-discovery.git"
REPO_DIR = "/content/camera-discovery"

from pathlib import Path
repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    !git clone -b "{REPO_BRANCH}" "{REPO_URL}" "{REPO_DIR}"
else:
    print(f"Repository already exists at {repo_dir}. Keeping existing checkout.")
%cd {REPO_DIR}
%pip install -e .[cloakbrowser] --no-build-isolation


## Credentials

In [ ]:
# Ollama Cloud / LLM credential setup.
# This avoids printing secrets. Configure OLLAMA_API_KEY in Colab: left sidebar > Secrets.
import os

try:
    from google.colab import userdata  # type: ignore
    OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
except Exception:
    OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY')

if OLLAMA_API_KEY:
    os.environ['OLLAMA_API_KEY'] = OLLAMA_API_KEY
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_PROVIDER', 'ollama-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_MODEL', 'gemma3:27b-cloud')
    print('Loaded OLLAMA_API_KEY from Colab userdata/environment')
else:
    print('OLLAMA_API_KEY not found. LLM-backed stages may fail unless another provider is configured.')

# Keep these visible so output records the provider/model, but never print the key.
print('LLM provider:', os.environ.get('CAMERA_DISCOVERY_LLM_PROVIDER', '(default from config)'))
print('LLM model:', os.environ.get('CAMERA_DISCOVERY_LLM_MODEL', '(default from config)'))


## Smoke tests

In [ ]:
# CLI and public import smoke tests.
import subprocess, sys

def run_cmd(cmd, *, env=None, check=True):
    print("\n$", " ".join(str(part) for part in cmd))
    result = subprocess.run([str(part) for part in cmd], env=env, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result

run_cmd(['camera-discovery', '--help'])
run_cmd(['camera-discovery', 'run', '--help'])
run_cmd(['camera-discovery', 'harvest-urls', '--help'])

from camera_discovery.services.discovery_engine import CandidateDiscoveryEngine
from camera_discovery.services.harvest_engine import CameraUrlHarvestEngine
import camera_discovery.cli
print('camera-discovery imports OK')

import camera_discovery
from camera_discovery.utils import geojson_viewer
print('camera_discovery package:', camera_discovery.__file__)
print('geojson_viewer module:', geojson_viewer.__file__)
print('target bbox overlay helper available:', hasattr(geojson_viewer, 'load_target_geometry_overlays'))


## Notebook-only helper functions

In [ ]:
# Notebook-only inspection helpers. These intentionally live in the notebook, not src/.
from __future__ import annotations

import json
import os
import shutil
import subprocess
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse


def read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path}")
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def iter_jsonl(path, limit=None):
    path = Path(path)
    if not path.exists():
        return
    with path.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if limit is not None and idx >= limit:
                break
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())


def top_hosts(path, url_field='url', limit=15):
    counts = Counter()
    for row in iter_jsonl(path):
        url = row.get(url_field) or row.get('stream_url') or row.get('source_url') or ''
        host = urlparse(url).netloc.casefold() or '(missing-host)'
        counts[host] += 1
    return counts.most_common(limit)


def media_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[row.get('media_type') or row.get('camera_type') or '(missing)'] += 1
    return dict(counts)


def scope_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[row.get('scope_status') or row.get('properties', {}).get('scope_status') or '(missing)'] += 1
    return dict(counts)


def print_json(path, keys=None):
    data = read_json(path)
    if data is None:
        return None
    if keys:
        data = {key: data.get(key) for key in keys}
    print(json.dumps(data, indent=2, sort_keys=True)[:12000])
    return data


def list_existing(paths):
    for path in paths:
        path = Path(path)
        print(f"{path}: {'exists' if path.exists() else 'missing'}" + (f" ({path.stat().st_size:,} bytes)" if path.exists() and path.is_file() else ''))


def package_output(output_dir, zip_name=None):
    output_dir = Path(output_dir)
    if zip_name is None:
        zip_name = str(output_dir).rstrip('/').replace('/', '_') + '.zip'
    zip_base = Path(zip_name).with_suffix('')
    archive = shutil.make_archive(str(zip_base), 'zip', root_dir=str(output_dir))
    print('Created archive:', archive)
    try:
        from google.colab import files  # type: ignore
        files.download(archive)
    except Exception:
        print('Download helper unavailable outside Colab. Archive remains at:', archive)
    return archive


def run_cli(cmd, *, env_overrides=None, check=True):
    env = os.environ.copy()
    if env_overrides:
        env.update({k: str(v) for k, v in env_overrides.items()})
    return run_cmd(cmd, env=env, check=check)


## Browser backend behavior

These notebooks default to browser capture disabled for structured endpoint / HLS workflows because the useful camera records usually come from static pages and JSON endpoints. Enable browser capture only when testing dynamic pages.

The cells below make the selected backend visible. They do not fake browser success.


In [ ]:
# Browser backend configuration for this notebook.
# For routine HLS/structured-endpoint tests, keep browser capture disabled.
BROWSER_BACKEND = "playwright"  # change to "cloakbrowser" when intentionally testing that backend
DISABLE_BROWSER_CAPTURE_ENV = {"CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE": "false", "CAMERA_DISCOVERY_BROWSER_BACKEND": BROWSER_BACKEND}
print('Browser backend selected:', BROWSER_BACKEND)
print('Browser capture default for this notebook:', DISABLE_BROWSER_CAPTURE_ENV['CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE'])
print('To test dynamic browser capture, remove --disable-browser-capture for harvest and set CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE=true for run.')


## Step 1 — Harvest all media types

In [ ]:
QUERY = "California traffic cameras"
HARVEST_DIR = Path('runs/harvest-california-all-media')
RERUN_HARVEST = False

expected = [HARVEST_DIR / 'harvest_summary.json', HARVEST_DIR / 'harvest_handoff.json', HARVEST_DIR / 'camera_urls.jsonl']
if RERUN_HARVEST and HARVEST_DIR.exists():
    shutil.rmtree(HARVEST_DIR)

if not all(path.exists() for path in expected):
    print('Running visible harvest CLI command. Output will stream below.')
    !camera-discovery harvest-urls "{QUERY}"       --output-dir "{HARVEST_DIR}"       --discovery-mode both       --max-search-queries 12       --max-search-results-per-query 25       --max-source-rows 1000       --max-pages-per-source 10       --max-urls 0       --media all       --disable-browser-capture       --progress-style plain
else:
    print('Skipping harvest: completion artifacts already exist. Set RERUN_HARVEST=True to rerun.')


## Inspect all-media harvest

In [ ]:
# Inspect harvest outputs.
HARVEST_DIR = Path(HARVEST_DIR)
print('Harvest directory:', HARVEST_DIR)
list_existing([
    HARVEST_DIR / 'harvest_summary.json',
    HARVEST_DIR / 'harvest_handoff.json',
    HARVEST_DIR / 'camera_urls.txt',
    HARVEST_DIR / 'camera_urls.csv',
    HARVEST_DIR / 'camera_urls.jsonl',
    HARVEST_DIR / 'camera_records.jsonl',
    HARVEST_DIR / 'camera_media_assets.jsonl',
    HARVEST_DIR / 'discovered_endpoints.jsonl',
    HARVEST_DIR / 'logs' / 'source_rows_summary.json',
    HARVEST_DIR / 'logs' / 'harvest_blind_search_diagnostics.jsonl',
    HARVEST_DIR / 'logs' / 'harvest_errors.jsonl',
])

summary = print_json(HARVEST_DIR / 'harvest_summary.json', keys=[
    'raw_count', 'unique_count', 'written_count', 'by_media_type', 'by_source_provider', 'by_source_host',
    'camera_record_count', 'media_asset_count', 'discovered_endpoint_count', 'warnings'
])
print('\nSource row summary:')
print_json(HARVEST_DIR / 'logs' / 'source_rows_summary.json')
print('\nHandoff manifest:')
print_json(HARVEST_DIR / 'harvest_handoff.json')
print('\nMedia counts from camera_urls.jsonl:', media_counts(HARVEST_DIR / 'camera_urls.jsonl'))
print('\nTop URL hosts from camera_urls.jsonl:', top_hosts(HARVEST_DIR / 'camera_urls.jsonl'))
print('\nJSONL counts:')
for name in ['camera_urls.jsonl', 'camera_records.jsonl', 'camera_media_assets.jsonl', 'discovered_endpoints.jsonl', 'harvest_camera_inventory.jsonl']:
    print(name, count_jsonl(HARVEST_DIR / name))
print('\nSample camera_urls.jsonl rows:')
for row in iter_jsonl(HARVEST_DIR / 'camera_urls.jsonl', limit=5):
    print(json.dumps(row, indent=2)[:2000])


## Step 2 — Run full validation from all-media handoff

In [ ]:
RUN_PROFILE = "balanced"
HTTP_TIMEOUT_SECONDS = 10
RUN_DIR = Path('runs/run-from-harvest-all-media-balanced')
RERUN_PIPELINE = False
expected_run = [RUN_DIR / 'logs' / 'run_summary.json', RUN_DIR / 'logs' / 'candidate_discovery_summary.json']
if RERUN_PIPELINE and RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

if not all(path.exists() for path in expected_run):
    print('Running visible CLI command. Output will stream below.')
    print('Validation is parallel and bounded by worker count; no validation candidate cap is applied.')
    print(f'Effective notebook profile: {RUN_PROFILE}; HTTP timeout: {HTTP_TIMEOUT_SECONDS}s')
    !camera-discovery run "{QUERY}"       --profile "{RUN_PROFILE}"       --output-dir "{RUN_DIR}"       --harvest-input "{HARVEST_DIR / 'harvest_handoff.json'}"       --harvest-input-mode handoff-only       --browser-backend "{BROWSER_BACKEND}"       --http-timeout "{HTTP_TIMEOUT_SECONDS}"       --progress-style plain
else:
    print('Skipping pipeline: completion artifacts already exist. Set RERUN_PIPELINE=True to rerun.')


## Inspect all-media pipeline outputs

## Target-resolution bbox/map overlay note

Target-resolution diagnostics now preserve the accepted Nominatim bbox and the effective bbox used downstream. For tiny precise targets, the effective bbox may be padded to the minimum practical extent; the original Nominatim bbox, padding reason, and minimum side length remain in `logs/target_resolution*.json`. Generated maps overlay target bounding boxes as border-only rectangles together with geocoder points and camera coordinate markers.


In [ ]:
# Inspect pipeline/run outputs.
RUN_DIR = Path(RUN_DIR)
print('Run directory:', RUN_DIR)
list_existing([
    RUN_DIR / 'logs' / 'run_summary.json',
    RUN_DIR / 'logs' / 'run_explanation.json',
    RUN_DIR / 'logs' / 'candidate_discovery_summary.json',
    RUN_DIR / 'logs' / 'candidate_priority_summary.json',
    RUN_DIR / 'logs' / 'validation_summary.json',
    RUN_DIR / 'logs' / 'validation_priority_summary.json',
    RUN_DIR / 'camera_candidates_table.csv',
    RUN_DIR / 'untrusted_camera_candidates.geojson',
    RUN_DIR / 'camera.geojson',
    RUN_DIR / 'map.html',
    RUN_DIR / 'review_artifacts.zip',
])

print('\nRun summary:')
print_json(RUN_DIR / 'logs' / 'run_summary.json')
print('\nCandidate discovery summary:')
print_json(RUN_DIR / 'logs' / 'candidate_discovery_summary.json')
print('\nCandidate priority summary:')
print_json(RUN_DIR / 'logs' / 'candidate_priority_summary.json')
print('\nValidation summary:')
print_json(RUN_DIR / 'logs' / 'validation_summary.json')

# Inspect CSV quickly without pandas.
csv_path = RUN_DIR / 'camera_candidates_table.csv'
if csv_path.exists():
    import csv
    with csv_path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    print('\nCandidate table rows:', len(rows))
    print('Media counts:', dict(Counter(row.get('camera_type') or row.get('media_type') or '(missing)' for row in rows)))
    print('Scope counts:', dict(Counter(row.get('scope_status') or '(missing)' for row in rows)))
    print('Priority buckets:', dict(Counter(row.get('candidate_priority_bucket') or '(missing)' for row in rows)))
    print('\nFirst 5 candidate table rows:')
    for row in rows[:5]:
        print({k: row.get(k) for k in ['camera_type', 'scope_status', 'candidate_priority_bucket', 'stream_url', 'latitude', 'longitude', 'validation_status'] if k in row})
else:
    print('No candidate table found.')

# GeoJSON feature counts.
for geojson_name in ['camera.geojson', 'untrusted_camera_candidates.geojson']:
    path = RUN_DIR / geojson_name
    data = read_json(path)
    if data:
        features = data.get('features', [])
        print(f"{geojson_name}: {len(features)} features")
        print('Feature scope counts:', dict(Counter((feat.get('properties') or {}).get('scope_status') or '(missing)' for feat in features)))

# Regenerate the map with the currently installed source code so reused run
# directories do not keep an older map.html without target bbox overlays.
from camera_discovery.utils.geojson_viewer import load_target_geometry_overlays, write_embedded_camera_map

target_overlays = load_target_geometry_overlays(RUN_DIR)
print('
Target bbox/point overlays discovered:', len(target_overlays))
if target_overlays:
    print('First target overlay:', {k: target_overlays[0].get(k) for k in ['target_label', 'bbox', 'effective_bbox', 'nominatim_bbox', 'lat', 'lon', 'geometry_source', 'bbox_padding_applied']})
    regenerated_map = write_embedded_camera_map(RUN_DIR, output_name='map.html')
    print('Regenerated map with target overlays:', regenerated_map)
    print('
Camera map status:')
    print_json(RUN_DIR / 'logs' / 'camera_map_status.json')
else:
    print('WARNING: no target-resolution bbox/point overlays found in logs/target_resolution*.json; map will show camera points only.')


## Non-HLS/non-image media review

In [ ]:
# Inspect harvested non-HLS media and how it appears downstream.
print('Harvest media counts:', media_counts(HARVEST_DIR / 'camera_urls.jsonl'))
print('Pipeline candidate table media counts are printed above if camera_candidates_table.csv exists.')
print('\nSample non-HLS harvest rows:')
shown = 0
for row in iter_jsonl(HARVEST_DIR / 'camera_urls.jsonl'):
    if row.get('media_type') != 'hls':
        print(json.dumps(row, indent=2)[:2000])
        shown += 1
    if shown >= 5:
        break
if shown == 0:
    print('No non-HLS rows found in final camera_urls.jsonl for this run.')


## Package outputs

In [ ]:
# Optional: package and download outputs.
# Run this after the workflow completes.
package_output('runs/run-from-harvest-all-media-full')


## Nominatim target geometry hierarchy inspection

This cell verifies that target maps prefer Nominatim polygon/multipolygon boundaries, fall back to the Nominatim rectangular bbox, and only use a generic padded bbox when no usable Nominatim geometry exists.


In [ ]:
# NOMINATIM_GEOMETRY_HIERARCHY_INSPECTION
from pathlib import Path
import json

run_dir = Path(globals().get("RUN_DIR", globals().get("OUTPUT_DIR", ".")))
if not (run_dir / "logs").exists() and Path("runs").exists():
    run_candidates = sorted(Path("runs").glob("*"), key=lambda p: p.stat().st_mtime if p.exists() else 0)
    if run_candidates:
        run_dir = run_candidates[-1]
target_file = run_dir / "logs" / "target_resolution_all.json"
if not target_file.exists():
    target_file = run_dir / "logs" / "target_resolution.json"
if target_file.exists():
    target_data = json.loads(target_file.read_text(encoding="utf-8"))
    targets = target_data.get("targets") if isinstance(target_data, dict) and isinstance(target_data.get("targets"), list) else [target_data]
    for target in targets:
        if not isinstance(target, dict):
            continue
        print("Target:", target.get("target_label") or target.get("canonical_target") or target.get("target_id"))
        print("  primary_geometry_source:", target.get("primary_geometry_source"))
        print("  has_target_geometry_geojson:", bool(target.get("target_geometry_geojson") or target.get("primary_geometry_geojson")))
        print("  fallback_geometry_source:", target.get("fallback_geometry_source"))
        print("  fallback_geometry_bbox:", target.get("fallback_geometry_bbox") or target.get("nominatim_bbox"))
        print("  last_fallback_geometry_source:", target.get("last_fallback_geometry_source"))
        print("  effective_bbox:", target.get("effective_bbox") or target.get("bbox"))
        print("  geocoder point:", (target.get("chosen_candidate") or {}).get("lat"), (target.get("chosen_candidate") or {}).get("lon"))
else:
    print("No target-resolution log found yet:", target_file)

try:
    from camera_discovery.utils.geojson_viewer import write_embedded_camera_map
    if target_file.exists():
        map_path = write_embedded_camera_map(run_dir)
        status_path = run_dir / "logs" / "camera_map_status.json"
        status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else {}
        print("Regenerated map:", map_path)
        print("Map primary geometry overlays:", status.get("target_primary_geometry_overlays"))
        print("Map fallback bbox overlays:", status.get("target_fallback_bbox_overlays"))
        print("Map last-fallback bbox overlays:", status.get("target_last_fallback_bbox_overlays"))
except Exception as exc:
    print("Map regeneration skipped/error:", repr(exc))


## Media validation dashboard and playlists
Inspect the new `media_validation_dashboard.json`, `playlists/`, and optional Google dorking summary artifacts when present.

In [ ]:
# Inspect media validation dashboard, playlists, and optional Google dorking counts
from pathlib import Path
import json, os

_run_dir_value = globals().get('OUTPUT_DIR') or globals().get('output_dir') or os.environ.get('CAMERA_DISCOVERY_OUTPUT_DIR') or 'runs/latest'
run_dir = Path(str(_run_dir_value))

dashboard_path = run_dir / 'media_validation_dashboard.json'
if dashboard_path.exists():
    print('media_validation_dashboard.json')
    print(json.dumps(json.loads(dashboard_path.read_text()), indent=2)[:4000])
else:
    print('media_validation_dashboard.json not found at', dashboard_path)

playlist_dir = run_dir / 'playlists'
if playlist_dir.exists():
    print('playlist artifacts:')
    for path in sorted(playlist_dir.glob('*')):
        print('-', path.relative_to(run_dir))
else:
    print('playlists/ not found at', playlist_dir)

dork_path = run_dir / 'logs' / 'google_dorking_summary.json'
if dork_path.exists():
    data = json.loads(dork_path.read_text())
    print('google_dorking:', {k: data.get(k) for k in ['enabled', 'queries_generated', 'results_seen', 'results_after_block_policy', 'promoted_source_leads', 'candidates_extracted']})
else:
    print('google_dorking summary not present')
